In [ ]:
%pip install selenium

In [1]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException
import time
from datetime import datetime
import re # For regular expressions to clean team names

In [ ]:
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd

# Define the URLs for the seasons you want to scrape
season_urls = {
    "2013-2014": "https://www.flashscore.es/futbol/inglaterra/premier-league-2013-2014/resultados/",
    "2014-2015": "https://www.flashscore.es/futbol/inglaterra/premier-league-2014-2015/resultados/",
    "2015-2016": "https://www.flashscore.es/futbol/inglaterra/premier-league-2015-2016/resultados/"
}

# Initialize the WebDriver outside the season loop
driver = webdriver.Chrome()

all_commentary_data = [] # This will store data from all seasons

for season, base_url in season_urls.items():
    print(f"\n--- Starting to scrape season: {season} ---")
    driver.get(base_url)

    # Step 1: Use Selenium to load all match links for the current season
    while True:
        try:
            # Wait for the "Mostrar más partidos" (show more) button to be clickable
            show_more = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="live-table"]/div[1]/div/div/a'))
            )
            driver.execute_script("arguments[0].click();", show_more)
            time.sleep(2)  # Give time for new content to load
        except (NoSuchElementException, TimeoutException):
            print(f"No more 'show more' button found or timed out for {season}.")
            break

    # Step 2: Extract match links and additional details (Home Team, Away Team, Date) for the current season
    current_season_match_details = []
    # Wait for some match elements to be present before finding all
    WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.XPATH, "//div[@class='event__match event__match--withRowLink event__match--static event__match--twoLine']"))
    )
    # Find all match rows which contain the link and participant/date info
    match_elements_rows = driver.find_elements(By.XPATH, "//div[@class='event__match event__match--withRowLink event__match--static event__match--twoLine']")

    for match_row_element in match_elements_rows:
        match_link = ""
        home_team = "N/A"
        away_team = "N/A"
        match_date = "N/A"

        try:
            # Extract the href from the 'a' tag within the current match row
            link_element = match_row_element.find_element(By.XPATH, ".//a[@class='eventRowLink']")
            href = link_element.get_attribute("href")
            if href and "/#/resumen-del-partido" in href:
                match_link = href
        except NoSuchElementException:
            pass # No link found in this row, skip or handle as needed

        if not match_link: # Only process if a valid link was found
            continue

        try:
            # Extract Home Team
            home_team_element = match_row_element.find_element(By.XPATH, ".//div[contains(@class, 'event__homeParticipant')]//*[contains(@class, 'wcl-name_3y6f5')]")
            home_team = home_team_element.text
        except NoSuchElementException:
            pass

        try:
            # Extract Away Team
            away_team_element = match_row_element.find_element(By.XPATH, ".//div[contains(@class, 'event__awayParticipant')]//*[contains(@class, 'wcl-name_3y6f5')]")
            away_team = away_team_element.text
        except NoSuchElementException:
            pass

        try:
            # Extract Match Date
            date_element = match_row_element.find_element(By.XPATH, ".//div[@class='event__time']")
            match_date = date_element.text
        except NoSuchElementException:
            pass

        current_season_match_details.append({
            "Season": season, # Add the season here
            "Match URL": match_link,
            "Home Team": home_team,
            "Away Team": away_team,
            "Match Date": match_date
        })

    print(f"Found {len(current_season_match_details)} match links with details for {season}.")

    # Step 3: Scrape commentaries using Selenium for the current season
    for i, match_info in enumerate(current_season_match_details):
        url = match_info["Match URL"]
        home_team = match_info["Home Team"]
        away_team = match_info["Away Team"]
        match_date = match_info["Match Date"]
        current_season_name = match_info["Season"] # Get the season name

        print(f"Processing match {i+1}/{len(current_season_match_details)} ({current_season_name}): {home_team} vs {away_team} ({match_date})")

        # Ensure the URL points to the commentary section
        if "#/resumen-del-partido/comentarios-en-directo" not in url:
            url_for_commentary = url.split('#')[0] + "#/resumen-del-partido/comentarios-en-directo/0"
        else:
            url_for_commentary = url

        current_match_commentary = {
            "Season": current_season_name, # Add season to each match entry
            "Match URL": url,
            "Home Team": home_team,
            "Away Team": away_team,
            "Match Date": match_date,
            "Commentary": "" # Initialize commentary field
        }

        try:
            driver.get(url_for_commentary)

            # Wait for the main commentary container to be present.
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.XPATH, "//div[@class='section liveCommentary']"))
            )
            time.sleep(2) # Give it a little more time to fully load, especially dynamic content

            # Try to find specific commentary entries using the provided class
            commentary_entries = driver.find_elements(By.XPATH, "//div[@class='section liveCommentary']//div[contains(@class, 'wcl-commentary_PDHM0')]")

            if commentary_entries:
                # Concatenate all text from individual commentary entries
                full_commentary_text = "\n".join([entry.text for entry in commentary_entries if entry.text.strip()])
                if full_commentary_text.strip():
                    current_match_commentary["Commentary"] = full_commentary_text
                else:
                    print(f"Found commentary containers but no actual commentary text extracted for {url}.")
                    current_match_commentary["Commentary"] = "Found containers but no detailed commentary text."
            else:
                # If no 'wcl-commentary_PDHM0' elements are found, check for a "no commentary" message
                try:
                    no_commentary_message = driver.find_element(By.XPATH, "//div[@class='section liveCommentary']//div[contains(text(), 'No hay comentarios')]")
                    commentary_message = no_commentary_message.text.strip()
                    print(f"No detailed commentary text found for {url}. Message: '{commentary_message}'")
                    current_match_commentary["Commentary"] = commentary_message
                except NoSuchElementException:
                    # If neither specific commentary entries nor a "no commentary" message are found,
                    # it's highly likely there's no commentary.
                    print(f"No specific commentary entries or 'no commentary' message found for {url}. Marking as 'No commentary available'.")
                    current_match_commentary["Commentary"] = "No commentary available for this match."
        except NoSuchElementException as e:
            print(f"Error finding main commentary container for {url_for_commentary}: {str(e)}. Skipping.")
            current_match_commentary["Commentary"] = f"Error: Main commentary container not found ({str(e)})."
        except TimeoutException:
            print(f"Timeout while loading commentary for {url_for_commentary}. Skipping.")
            current_match_commentary["Commentary"] = "Timeout: Could not load commentary page or find main commentary container."
        except Exception as e:
            print(f"An unexpected error occurred processing {url_for_commentary}: {str(e)}. Skipping.")
            current_match_commentary["Commentary"] = f"Unexpected Error: {str(e)}"

        all_commentary_data.append(current_match_commentary)

driver.quit()  # Close Selenium browser after all scraping

# Step 4: Save to CSV or analyze
df = pd.DataFrame(all_commentary_data)
output_filename = "flashscore_commentary_2013-2016.csv"
df.to_csv(output_filename, index=False)
print(f"\nDone: commentary for seasons 2013-14 to 2015-16 saved to {output_filename}")